In [ ]:
!pip install transformers scikit-learn accelerate -q

import torch, json, os, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import roc_auc_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {device}')

# KatFish 데이터 로드
if not os.path.exists('katfishnet'):
    !git clone https://github.com/Shinwoo-Park/detecting_llm_generated_korean_text_through_linguistic_analysis.git katfishnet

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

essay    = load_jsonl('katfishnet/katfish_dataset/essay.jsonl')
abstract = load_jsonl('katfishnet/katfish_dataset/abstract.jsonl')
poetry   = load_jsonl('katfishnet/katfish_dataset/poetry.jsonl')

df_essay    = pd.DataFrame(essay);    df_essay['genre']    = 'essay'
df_abstract = pd.DataFrame(abstract); df_abstract['genre'] = 'abstract'
df_poetry   = pd.DataFrame(poetry);   df_poetry['genre']   = 'poetry'
df_all = pd.concat([df_essay, df_abstract, df_poetry]).reset_index(drop=True)

print(f"전체: {len(df_all)}개")
print(df_all['written_by'].value_counts())

In [ ]:
def compute_binoculars(text, m_obs, tok_obs, m_per, tok_per, device, max_length=256):
    inp_o = tok_obs(text, return_tensors='pt', truncation=True, max_length=max_length).to(device)
    with torch.no_grad():
        loss_o = m_obs(**inp_o, labels=inp_o['input_ids']).loss.item()

    inp_p  = tok_per(text, return_tensors='pt', truncation=True, max_length=max_length).to(device)
    inp_o2 = tok_obs(text, return_tensors='pt', truncation=True, max_length=max_length).to(device)
    with torch.no_grad():
        logits_p = m_per(**inp_p).logits
        logits_o = m_obs(**inp_o2).logits

    seq_len = min(logits_p.shape[1], logits_o.shape[1]) - 1
    if seq_len <= 0:
        return 0.0

    lp = logits_p[:, :seq_len, :]
    lo = logits_o[:, :seq_len, :]
    v  = min(lp.shape[-1], lo.shape[-1])
    lp, lo = lp[..., :v], lo[..., :v]

    cross_loss = -(torch.softmax(lp, dim=-1) * torch.log_softmax(lo, dim=-1)).sum(-1).mean().item()
    return loss_o / (cross_loss + 1e-10)

# 모델 로드
print("Polyglot-Ko-1.3B 로드 중...")
tok1 = AutoTokenizer.from_pretrained('EleutherAI/polyglot-ko-1.3b')
m1   = AutoModelForCausalLM.from_pretrained('EleutherAI/polyglot-ko-1.3b').eval().to(device)

print("Polyglot-Ko-3.8B 로드 중...")
tok2 = AutoTokenizer.from_pretrained('EleutherAI/polyglot-ko-3.8b')
m2   = AutoModelForCausalLM.from_pretrained('EleutherAI/polyglot-ko-3.8b').eval().to(device)

print("완료!")

In [ ]:
# Human 20% 테스트셋 고정
human_df   = df_all[df_all['label'] == 0]
human_test = human_df.sample(frac=0.2, random_state=42)
print(f"Human 테스트셋: {len(human_test)}개")

# LLM별 그룹 (GPT-4o 제외)
llm_groups = {
    'Solar':    df_all[df_all['written_by'].str.contains('solar',  case=False, na=False)],
    'Qwen2':    df_all[df_all['written_by'].str.contains('qwen',   case=False, na=False)],
    'Llama3.1': df_all[df_all['written_by'].str.contains('llama',  case=False, na=False)],
}
for name, grp in llm_groups.items():
    print(f"{name}: {len(grp)}개")

거짓 양성률 체크

In [ ]:
from sklearn.metrics import roc_auc_score, recall_score
import numpy as np
from tqdm import tqdm

# 임계값 설정 함수
def find_threshold(scores, labels, target_fpr=0.05):
    """FPR <= target_fpr 조건에서 최대 Recall을 주는 임계값 탐색"""
    thresholds = np.linspace(scores.min(), scores.max(), 1000)
    best_thresh = None
    best_recall = 0
    for t in thresholds:
        preds = (scores >= t).astype(int)
        tn = ((preds == 0) & (labels == 0)).sum()
        fp = ((preds == 1) & (labels == 0)).sum()
        tp = ((preds == 1) & (labels == 1)).sum()
        fn = ((preds == 0) & (labels == 1)).sum()
        fpr = fp / (fp + tn + 1e-10)
        recall = tp / (tp + fn + 1e-10)
        if fpr <= target_fpr and recall > best_recall:
            best_recall = recall
            best_thresh = t
    return best_thresh, best_recall

# ============================================================
# Ko-Binoculars 1.3b + 3.8b OOD 실험 (AUC + Recall + FPR)
# ============================================================

print("Human 점수 계산 중...")
human_scores = []
for text in tqdm(human_test['text'].tolist()):
    human_scores.append(compute_binoculars(text, m1, tok1, m2, tok2, device))
human_scores = np.array(human_scores)

results_full = {}

for llm_name, llm_df in llm_groups.items():
    print(f"\n{llm_name} 점수 계산 중...")
    llm_scores = []
    for text in tqdm(llm_df['text'].tolist()):
        llm_scores.append(compute_binoculars(text, m1, tok1, m2, tok2, device))
    llm_scores = np.array(llm_scores)

    scores_all = np.concatenate([human_scores, llm_scores])
    labels_all = np.array([0]*len(human_scores) + [1]*len(llm_scores))

    # Binoculars: 낮을수록 AI → 반전
    scores_for_eval = -scores_all

    auc = roc_auc_score(labels_all, scores_for_eval)

    # FPR=5% 조건에서 Recall
    thresh, recall_at_5fpr = find_threshold(scores_for_eval, labels_all, target_fpr=0.05)

    # FPR=10% 조건에서 Recall
    _, recall_at_10fpr = find_threshold(scores_for_eval, labels_all, target_fpr=0.10)

    # 전체 FPR (임계값 0.5 기준 - AUC 최적 임계값)
    if thresh is not None:
        preds = (scores_for_eval >= thresh).astype(int)
        fp = ((preds==1) & (labels_all==0)).sum()
        tn = ((preds==0) & (labels_all==0)).sum()
        fpr_at_thresh = fp / (fp + tn + 1e-10)
    else:
        fpr_at_thresh = None

    results_full[llm_name] = {
        'auc': auc,
        'recall@fpr5': recall_at_5fpr,
        'recall@fpr10': recall_at_10fpr,
        'fpr': fpr_at_thresh
    }

# 결과 출력
print("\n===== Ko-Binoculars (1.3b+3.8b) 전체 결과 =====")
print(f"{'LLM':<12} {'AUC':>8} {'Recall@FPR5%':>14} {'Recall@FPR10%':>15} {'FPR(thresh)':>13}")
print("-" * 65)
for llm_name, res in results_full.items():
    print(f"{llm_name:<12} "
          f"{res['auc']*100:>8.2f} "
          f"{res['recall@fpr5']*100:>14.2f} "
          f"{res['recall@fpr10']*100:>15.2f} "
          f"{res['fpr']*100 if res['fpr'] else 0:>13.2f}")

aucs = [r['auc'] for r in results_full.values()]
r5s  = [r['recall@fpr5'] for r in results_full.values()]
r10s = [r['recall@fpr10'] for r in results_full.values()]
print("-" * 65)
print(f"{'평균':<12} "
      f"{np.mean(aucs)*100:>8.2f} "
      f"{np.mean(r5s)*100:>14.2f} "
      f"{np.mean(r10s)*100:>15.2f}")

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np
from tqdm import tqdm

def find_threshold(scores, labels, target_fpr=0.05):
    thresholds = np.linspace(scores.min(), scores.max(), 1000)
    best_thresh, best_recall = None, 0
    for t in thresholds:
        preds = (scores >= t).astype(int)
        fp = ((preds==1) & (labels==0)).sum()
        tn = ((preds==0) & (labels==0)).sum()
        tp = ((preds==1) & (labels==1)).sum()
        fn = ((preds==0) & (labels==1)).sum()
        fpr = fp / (fp + tn + 1e-10)
        recall = tp / (tp + fn + 1e-10)
        if fpr <= target_fpr and recall > best_recall:
            best_recall = recall
            best_thresh = t
    return best_thresh, best_recall

# 세 쌍 정의
pairs = {
    '1.3b+1.3b': (m1, tok1, m1, tok1),
    '1.3b+3.8b': (m1, tok1, m2, tok2),
    '3.8b+1.3b': (m2, tok2, m1, tok1),
}

# 모든 쌍의 점수 저장
all_pair_scores = {}  # {pair_name: {llm: scores, 'human': scores}}

for pair_name, (m_obs, tok_obs, m_per, tok_per) in pairs.items():
    print(f"\n===== {pair_name} =====")
    all_pair_scores[pair_name] = {}

    # Human 점수
    print("  Human 계산 중...")
    h_sc = []
    for text in tqdm(human_test['text'].tolist()):
        h_sc.append(compute_binoculars(text, m_obs, tok_obs, m_per, tok_per, device))
    all_pair_scores[pair_name]['human'] = np.array(h_sc)

    # LLM별 점수
    for llm_name, llm_df in llm_groups.items():
        print(f"  {llm_name} 계산 중...")
        l_sc = []
        for text in tqdm(llm_df['text'].tolist()):
            l_sc.append(compute_binoculars(text, m_obs, tok_obs, m_per, tok_per, device))
        all_pair_scores[pair_name][llm_name] = np.array(l_sc)

print("\n모든 점수 계산 완료!")

# ============================================================
# 각 쌍 단독 결과 + 앙상블 결과
# ============================================================

def evaluate(h_scores, l_scores, label=''):
    scores = np.concatenate([h_scores, l_scores])
    labels = np.array([0]*len(h_scores) + [1]*len(l_scores))
    scores_eval = -scores  # 낮을수록 AI → 반전
    auc = roc_auc_score(labels, scores_eval)
    _, r5  = find_threshold(scores_eval, labels, target_fpr=0.05)
    _, r10 = find_threshold(scores_eval, labels, target_fpr=0.10)
    return auc*100, r5*100, r10*100

print("\n===== 단독 쌍 결과 =====")
print(f"{'방법':<20} {'→Solar':>8} {'→Qwen2':>8} {'→Llama3.1':>10} {'평균AUC':>9} {'R@FPR5':>8} {'R@FPR10':>9}")
print("-" * 80)

for pair_name in pairs.keys():
    aucs, r5s, r10s = [], [], []
    for llm_name in llm_groups.keys():
        h = all_pair_scores[pair_name]['human']
        l = all_pair_scores[pair_name][llm_name]
        auc, r5, r10 = evaluate(h, l)
        aucs.append(auc); r5s.append(r5); r10s.append(r10)
    print(f"{pair_name:<20} "
          f"{aucs[0]:>8.2f} {aucs[1]:>8.2f} {aucs[2]:>10.2f} "
          f"{np.mean(aucs):>9.2f} {np.mean(r5s):>8.2f} {np.mean(r10s):>9.2f}")

# 앙상블
print("\n===== 앙상블 결과 (세 쌍 평균) =====")
print(f"{'방법':<20} {'→Solar':>8} {'→Qwen2':>8} {'→Llama3.1':>10} {'평균AUC':>9} {'R@FPR5':>8} {'R@FPR10':>9}")
print("-" * 80)

aucs, r5s, r10s = [], [], []
for llm_name in llm_groups.keys():
    h_ens = np.mean([all_pair_scores[p]['human']     for p in pairs], axis=0)
    l_ens = np.mean([all_pair_scores[p][llm_name]   for p in pairs], axis=0)
    auc, r5, r10 = evaluate(h_ens, l_ens)
    aucs.append(auc); r5s.append(r5); r10s.append(r10)

print(f"{'앙상블 (1.3+1.3+3.8)':<20} "
      f"{aucs[0]:>8.2f} {aucs[1]:>8.2f} {aucs[2]:>10.2f} "
      f"{np.mean(aucs):>9.2f} {np.mean(r5s):>8.2f} {np.mean(r10s):>9.2f}")

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.free',
                       '--format=csv,noheader,nounits'], capture_output=True, text=True)
used, free = result.stdout.strip().split(', ')
print(f"GPU 메모리 - 사용중: {int(used)/1024:.1f}GB / 여유: {int(free)/1024:.1f}GB")

In [ ]:
# m2가 이미 로드되어 있으니까 바로 실행
print("3.8b + 3.8b 실험 시작...")

human_scores_38 = []
for text in tqdm(human_test['text'].tolist(), desc='Human'):
    human_scores_38.append(compute_binoculars(text, m2, tok2, m2, tok2, device))
human_scores_38 = np.array(human_scores_38)

results_38 = {}
for llm_name, llm_df in llm_groups.items():
    print(f"{llm_name} 계산 중...")
    l_sc = []
    for text in tqdm(llm_df['text'].tolist()):
        l_sc.append(compute_binoculars(text, m2, tok2, m2, tok2, device))
    l_sc = np.array(l_sc)

    scores_all = np.concatenate([human_scores_38, l_sc])
    labels_all = np.array([0]*len(human_scores_38) + [1]*len(l_sc))
    scores_eval = -scores_all

    auc = roc_auc_score(labels_all, scores_eval)
    _, r5  = find_threshold(scores_eval, labels_all, target_fpr=0.05)
    _, r10 = find_threshold(scores_eval, labels_all, target_fpr=0.10)
    results_38[llm_name] = {'auc': auc, 'r5': r5, 'r10': r10}

print("\n===== 3.8b + 3.8b 결과 =====")
print(f"{'LLM':<12} {'AUC':>8} {'R@FPR5%':>10} {'R@FPR10%':>10}")
print("-" * 45)
for llm_name, res in results_38.items():
    print(f"{llm_name:<12} {res['auc']*100:>8.2f} {res['r5']*100:>10.2f} {res['r10']*100:>10.2f}")

aucs = [r['auc'] for r in results_38.values()]
r5s  = [r['r5']  for r in results_38.values()]
r10s = [r['r10'] for r in results_38.values()]
print("-" * 45)
print(f"{'평균':<12} {np.mean(aucs)*100:>8.2f} {np.mean(r5s)*100:>10.2f} {np.mean(r10s)*100:>10.2f}")

print("\n===== 전체 비교 =====")
print(f"{'모델 쌍':<20} {'AUC':>8} {'R@FPR5%':>10} {'R@FPR10%':>10}")
print("-" * 50)
prev = [('1.3b+1.3b', 61.51, 37.58, 40.40),
        ('1.3b+3.8b', 65.27, 32.13, 42.02),
        ('3.8b+1.3b', 61.67, 34.59, 38.94),
        ('앙상블',    62.87, 35.24, 40.31)]
for name, auc, r5, r10 in prev:
    print(f"{name:<20} {auc:>8.2f} {r5:>10.2f} {r10:>10.2f}")
print(f"{'3.8b+3.8b':<20} {np.mean(aucs)*100:>8.2f} {np.mean(r5s)*100:>10.2f} {np.mean(r10s)*100:>10.2f}")

모델 사이즈에 따른 성능 차이 확인하기 same, big-small, small-big

In [ ]:
# ============================================================
# Ablation: 모델 쌍 크기 조합별 OOD 실험
# m1 (1.3b), m2 (3.8b) 이미 로드되어 있다고 가정
# ============================================================

ablation_pairs = {
    '1.3b + 1.3b': (m1, tok1, m1, tok1),
    '1.3b + 3.8b': (m1, tok1, m2, tok2),
    '3.8b + 1.3b': (m2, tok2, m1, tok1),
}

ablation_ood_results = {}

for pair_name, (m_obs, tok_obs, m_per, tok_per) in ablation_pairs.items():
    print(f"\n===== {pair_name} =====")

    # Human 점수 계산
    h_scores = []
    for text in tqdm(human_test['text'].tolist(), desc='Human'):
        h_scores.append(compute_binoculars(text, m_obs, tok_obs, m_per, tok_per, device))
    h_scores = np.array(h_scores)

    # LLM별 점수 계산
    pair_results = {}
    for llm_name, llm_df in llm_groups.items():
        l_scores = []
        for text in tqdm(llm_df['text'].tolist(), desc=llm_name):
            l_scores.append(compute_binoculars(text, m_obs, tok_obs, m_per, tok_per, device))
        l_scores = np.array(l_scores)

        scores_all = np.concatenate([h_scores, l_scores])
        labels_all = np.array([0]*len(h_scores) + [1]*len(l_scores))
        auc = roc_auc_score(labels_all, -scores_all)
        pair_results[llm_name] = auc

    ablation_ood_results[pair_name] = pair_results
    avg = np.mean(list(pair_results.values()))
    print(f"  Solar: {pair_results['Solar']*100:.2f} | Qwen2: {pair_results['Qwen2']*100:.2f} | Llama3.1: {pair_results['Llama3.1']*100:.2f} | 평균: {avg*100:.2f}")

# 최종 결과표
print("\n===== Ablation 결과표 (OOD 방식) =====")
print(f"{'모델 쌍':<20} {'→Solar':>8} {'→Qwen2':>8} {'→Llama3.1':>10} {'평균':>8}")
print("-" * 55)
for pair_name, results in ablation_ood_results.items():
    avg = np.mean(list(results.values())) * 100
    marker = ' ← 최고' if avg == max(np.mean(list(r.values()))*100 for r in ablation_ood_results.values()) else ''
    print(f"{pair_name:<20} {results['Solar']*100:>8.2f} {results['Qwen2']*100:>8.2f} {results['Llama3.1']*100:>10.2f} {avg:>8.2f}{marker}")

In [ ]:
# ============================================================
# 최적화: Human 점수를 모델 쌍별로 한 번만 계산
# ============================================================

ablation_genre_results = {}

for pair_name, (m_obs, tok_obs, m_per, tok_per) in ablation_pairs.items():
    print(f"\n===== {pair_name} =====")
    ablation_genre_results[pair_name] = {}

    # Human 점수 한 번만 계산
    print("  Human 점수 계산 중...")
    h_scores_dict = {}  # 텍스트 → 점수 딕셔너리로 저장
    for text in tqdm(human_test['text'].tolist()):
        h_scores_dict[text] = compute_binoculars(
            text, m_obs, tok_obs, m_per, tok_per, device)

    for genre in ['essay', 'abstract', 'poetry']:
        ablation_genre_results[pair_name][genre] = {}

        h_genre = human_test[human_test['genre'] == genre]
        h_sc    = np.array([h_scores_dict[t] for t in h_genre['text'].tolist()])

        for llm_name, llm_df in llm_groups.items():
            l_genre = llm_df[llm_df['genre'] == genre]
            if len(l_genre) == 0:
                continue

            l_sc = []
            for text in tqdm(l_genre['text'].tolist(), desc=f'{llm_name}-{genre}'):
                l_sc.append(compute_binoculars(
                    text, m_obs, tok_obs, m_per, tok_per, device))
            l_sc = np.array(l_sc)

            sc_all = np.concatenate([h_sc, l_sc])
            lb_all = np.array([0]*len(h_sc) + [1]*len(l_sc))
            if len(set(lb_all)) == 2:
                ablation_genre_results[pair_name][genre][llm_name] = \
                    roc_auc_score(lb_all, -sc_all) * 100

# 결과 출력
for genre in ['essay', 'abstract', 'poetry']:
    print(f"\n===== {genre.upper()} =====")
    print(f"{'모델 쌍':<20} {'→Solar':>8} {'→Qwen2':>8} {'→Llama3.1':>10} {'평균':>8}")
    print("-" * 55)
    for pair_name in ablation_pairs.keys():
        vals = ablation_genre_results[pair_name][genre]
        scores = list(vals.values())
        avg = np.mean(scores) if scores else 0
        marker = ' ←' if avg == max(
            np.mean(list(ablation_genre_results[p][genre].values()))
            for p in ablation_pairs.keys()
        ) else ''
        print(f"{pair_name:<20} "
              f"{vals.get('Solar', 0):>8.2f} "
              f"{vals.get('Qwen2', 0):>8.2f} "
              f"{vals.get('Llama3.1', 0):>10.2f} "
              f"{avg:>8.2f}{marker}")

---

In [ ]:
# ============================================================
# OOD 실험 실행
# ============================================================

# 먼저 human_test에 대한 점수 계산
print("Human 테스트셋 점수 계산 중...")
human_scores = []
for text in tqdm(human_test['text'].tolist()):
    human_scores.append(compute_binoculars(text, m1, tok1, m2, tok2, device))
human_scores = np.array(human_scores)

# LLM별 OOD 테스트
ood_results = {}
genre_ood_results = {}

for llm_name, llm_df in llm_groups.items():
    print(f"\n{llm_name} 점수 계산 중...")
    llm_scores = []
    for text in tqdm(llm_df['text'].tolist()):
        llm_scores.append(compute_binoculars(text, m1, tok1, m2, tok2, device))
    llm_scores = np.array(llm_scores)

    # 전체 AUC
    scores_combined = np.concatenate([human_scores, llm_scores])
    labels_combined = np.array([0]*len(human_scores) + [1]*len(llm_scores))
    auc = roc_auc_score(labels_combined, -scores_combined)
    ood_results[llm_name] = auc

    # 장르별 AUC
    genre_ood_results[llm_name] = {}
    for genre in ['essay', 'abstract', 'poetry']:
        h_genre = human_test[human_test['genre'] == genre]
        l_genre = llm_df[llm_df['genre'] == genre]

        if len(h_genre) == 0 or len(l_genre) == 0:
            continue

        h_idx = [i for i, t in enumerate(human_test['text'].tolist())
                 if t in set(h_genre['text'].tolist())]
        h_sc  = human_scores[h_idx]

        l_sc = []
        for text in l_genre['text'].tolist():
            l_sc.append(compute_binoculars(text, m1, tok1, m2, tok2, device))
        l_sc = np.array(l_sc)

        sc_all = np.concatenate([h_sc, l_sc])
        lb_all = np.array([0]*len(h_sc) + [1]*len(l_sc))

        if len(set(lb_all)) == 2:
            genre_ood_results[llm_name][genre] = roc_auc_score(lb_all, -sc_all) * 100

print("\n계산 완료!")

In [ ]:
# ============================================================
# 결과 출력 (KatFishNet Table 3 형식)
# ============================================================

genres = ['essay', 'abstract', 'poetry']

print("===== Ko-Binoculars OOD 실험 결과 (KatFishNet 방식) =====")
print(f"{'방법':<20} {'→Solar':>8} {'→Qwen2':>8} {'→Llama3.1':>10} {'평균':>8}")
print("-" * 55)

# 장르별 출력
for genre in genres:
    scores_per_llm = []
    row = f"Ko-Bino ({genre:<8})"
    for llm in ['Solar', 'Qwen2', 'Llama3.1']:
        val = genre_ood_results.get(llm, {}).get(genre, None)
        if val is not None:
            row += f" {val:>8.2f}"
            scores_per_llm.append(val)
        else:
            row += f" {'  -':>8}"
    if scores_per_llm:
        row += f" {np.mean(scores_per_llm):>8.2f}"
    print(row)

# 전체 평균
print("-" * 55)
all_aucs = [v*100 for v in ood_results.values()]
print(f"{'Ko-Bino (전체)':<20}", end="")
for llm in ['Solar', 'Qwen2', 'Llama3.1']:
    print(f" {ood_results.get(llm, 0)*100:>8.2f}", end="")
print(f" {np.mean(all_aucs):>8.2f}")

print()
print("===== KatFishNet 논문 Table 3 비교 =====")
print(f"{'방법':<25} {'→Solar':>8} {'→Qwen2':>8} {'→Llama3.1':>10} {'평균':>8}")
print("-" * 60)
baselines = [
    ('DetectGPT',          67.04, 64.00, 67.02, 66.02),
    ('LLM Paraphrasing',   71.32, 58.79, 61.51, 63.87),
    ('KatFishNet (Punct.)',  62.65, 93.45, 63.22, 73.10),  # poetry 기준
    ('Ko-Binoculars(ours)', *[ood_results.get(l,0)*100
                               for l in ['Solar','Qwen2','Llama3.1']],
                             np.mean(all_aucs)),
]
for row in baselines:
    name = row[0]
    vals = row[1:]
    print(f"{name:<25}", end="")
    for v in vals:
        print(f" {v:>8.2f}", end="")
    print()

print("\n* KatFishNet 수치는 Park et al. (ACL 2025) Table 3 Poetry 행에서 인용")
print("* Ko-Binoculars는 Human 20% 테스트셋 + LLM 전체로 OOD 평가")